###VACUUM COMMAND IN DATABRICKS (DELTA LAKE)
####PURPOSE:
- The VACUUM command in Databricks deletes obsolete files from a Delta table to free up storage.
- Obsolete files are created during operations like UPDATE, DELETE, MERGE, or OPTIMIZE.
####WHY NEEDED:
- Delta tables maintain a transaction log (.delta_log) that keeps old versions of files
- for time-travel and ACID compliance. Over time, these files accumulate and occupy storage.
- VACUUM safely removes files older than the retention period.
####COMMAND SYNTAX:
 VACUUM table_name [RETAIN num HOURS]
####DEFAULT RETENTION:
- By default, Delta enforces a 7-day retention period to prevent accidental data loss.
- To override, you can specify a shorter period using RETAIN num HOURS.
####NOTES AND BEST PRACTICES:
- Do not reduce retention below 1 hour without understanding consequences.
- Always use VACUUM after OPTIMIZE to clean obsolete files if needed.
- VACUUM only affects physical files, not the transaction log or metadata.

In [0]:
%sql
-- Switch to the target catalog and schema
USE new_catalog.default_schema;

-- Create table only if it does not already exist
-- Table: product_inventory
-- Purpose: Maintain product details and inventory levels
-- Format: Delta Lake for ACID transactions, scalability, and time travel

CREATE TABLE IF NOT EXISTS product_inventory (
    product_id INT,          -- Unique product identifier
    product_name STRING,     -- Name of the product
    category STRING,         -- Product category (e.g., electronics, apparel)
    price DOUBLE,            -- Unit price of the product
    quantity INT,            -- Available stock quantity
    updated_date DATE        -- Last updated date for inventory record
)
USING DELTA;                 -- Delta format ensures reliability and performance

In [0]:
%sql
-- Step 2: Insert initial data into product_inventory
-- Each INSERT adds rows to the Delta table.
-- This simulates starting inventory levels for different product categories.

INSERT INTO product_inventory VALUES
 (1, 'Laptop', 'Electronics', 65000, 10, '2025-10-01'),   -- High-value electronics item
 (2, 'Headphones', 'Electronics', 2500, 50, '2025-10-01'), -- Accessories with larger stock
 (3, 'Desk Chair', 'Furniture', 4500, 20, '2025-10-01');   -- Furniture item with moderate stock

In [0]:
%sql
-- Step 3: Update some records
-- This operation modifies rows in the product_inventory table.
-- Delta Lake will create new Parquet file versions to reflect these changes.
-- Old versions remain accessible via time travel.

UPDATE product_inventory
SET price = price * 1.1,       -- Apply a 10% price increase
    updated_date = '2025-10-05' -- Update the last modified date
WHERE category = 'Electronics'; -- Only update products in the Electronics category

In [0]:
%sql
-- Step 4: View Delta table history
-- DESCRIBE HISTORY displays the transaction log for the Delta table.
-- It shows all operations such as CREATE, INSERT, UPDATE, DELETE, OPTIMIZE.
-- Each entry includes user, timestamp, operation type, and version number.
-- This is essential for auditing, debugging, and time travel queries.

DESCRIBE HISTORY product_inventory;

In [0]:
%sql
-- Step 5: View current table details
-- DESCRIBE DETAIL returns metadata about the Delta table.
-- Includes: catalog, schema, table type, provider, location,
-- row count, file count, size in bytes, and last modification time.
-- Useful for auditing, monitoring storage growth, and performance tuning.

DESCRIBE DETAIL product_inventory;

In [0]:
%sql
-- Disable retention duration check (for demo only)
-- Default retention is 7 days for safety
-- Run VACUUM to permanently delete obsolete files
-- WARNING: Irreversible! Use only for demo or test

--SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM product_inventory RETAIN 0 HOURS;